# 03 Data Cleaning

Notebook ini membersihkan 8 dataset raw secara terpisah, menstandarkan kolom penting, menyatukan format tanggal/amount/kategori, dan menyediakan dataframe akhir di memory.

Output utama di memory:
- `cleaned_datasets`: dictionary dataframe clean per dataset.
- `cleaning_summary_df`: ringkasan hasil cleaning.
- `validation_df`: hasil validasi kualitas cleaning.
- `df_cleaned`: gabungan clean untuk tahap berikutnya.


## 1. Import Library


In [1]:
from pathlib import Path
import warnings
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from fingo_ds1.config import RAW_DATA_PATH
except ImportError:
    RAW_DATA_PATH = None


## 2. Konfigurasi Path dan Tampilan


In [2]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if RAW_DATA_PATH is None:
    RAW_DATA_PATH = project_root / 'data' / 'raw'

raw_data_path = Path(RAW_DATA_PATH)

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

print(f'Project root  : {project_root}')
print(f'Raw data path : {raw_data_path}')
print('Mode          : display-only. Tidak ada file output yang disimpan.')


Project root  : /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Raw data path : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw
Mode          : display-only. Tidak ada file output yang disimpan.


## 3. Dataset Catalog


In [3]:
DATASET_CATALOG = [
    {
        'dataset_id': 'ecommerce_2024_01_january',
        'dataset_name': 'E-Commerce Sales - January 2024',
        'domain': 'ecommerce_sales',
        'period': '2024-01',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '01_JanuarySales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_06_june',
        'dataset_name': 'E-Commerce Sales - June 2024',
        'domain': 'ecommerce_sales',
        'period': '2024-06',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '02_JuneSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_12_december',
        'dataset_name': 'E-Commerce Sales - December 2024',
        'domain': 'ecommerce_sales',
        'period': '2024-12',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '03_DecemberSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_02_february',
        'dataset_name': 'E-Commerce Sales - February 2025',
        'domain': 'ecommerce_sales',
        'period': '2025-02',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '04_FebruarySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_07_july',
        'dataset_name': 'E-Commerce Sales - July 2025',
        'domain': 'ecommerce_sales',
        'period': '2025-07',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '05_JulySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_11_november',
        'dataset_name': 'E-Commerce Sales - November 2025',
        'domain': 'ecommerce_sales',
        'period': '2025-11',
        'path': raw_data_path / 'Indonesian_Ecommerce_sales' / '06_NovemberSales2025_clean.xlsx',
    },
    {
        'dataset_id': 'daily_household_transactions',
        'dataset_name': 'Daily Household Transactions',
        'domain': 'household_finance',
        'period': None,
        'path': raw_data_path / 'daily_household_transaction' / 'Daily Household Transactions.csv',
    },
    {
        'dataset_id': 'personal_finance',
        'dataset_name': 'Personal Finance Dataset',
        'domain': 'personal_finance',
        'period': None,
        'path': raw_data_path / 'personal_finance' / 'Personal_Finance_Dataset.csv',
    },
]

assert len(DATASET_CATALOG) == 8, 'Catalog harus berisi tepat 8 dataset.'


## 4. Validasi Dataset Catalog


In [4]:
def build_catalog_frame(catalog):
    return pd.DataFrame(
        {
            **item,
            'path': str(item['path'].relative_to(project_root)),
            'exists': item['path'].exists(),
        }
        for item in catalog
    )


catalog_df = build_catalog_frame(DATASET_CATALOG)
display(catalog_df)

if not catalog_df['exists'].all():
    missing = catalog_df.loc[~catalog_df['exists'], 'path'].tolist()
    raise FileNotFoundError(f'File raw belum lengkap: {missing}')


,dataset_id,dataset_name,domain,period,path,exists
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,True
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,True
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,True
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,True
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,True
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,True
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,True
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,True


## 5. Reusable Loader


In [5]:
EXCEL_NAMESPACE = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
SUPPORTED_EXTENSIONS = {'.csv', '.xlsx', '.xls'}


def make_unique_columns(columns):
    counts = {}
    result = []

    for column in columns:
        name = str(column).strip() if pd.notna(column) else 'unnamed_column'
        name = name or 'unnamed_column'
        counts[name] = counts.get(name, 0) + 1
        result.append(name if counts[name] == 1 else f'{name}_{counts[name]}')

    return result


def excel_column_index(cell_reference):
    letters = ''.join(character for character in str(cell_reference) if character.isalpha()) or 'A'
    index = 0

    for letter in letters.upper():
        index = index * 26 + ord(letter) - ord('A') + 1

    return index - 1


def read_excel_without_engine(path):
    with ZipFile(path) as archive:
        names = set(archive.namelist())
        shared_strings = []

        if 'xl/sharedStrings.xml' in names:
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            shared_strings = [
                ''.join(text.text or '' for text in item.findall('.//main:t', EXCEL_NAMESPACE))
                for item in root.findall('main:si', EXCEL_NAMESPACE)
            ]

        worksheet_names = sorted(name for name in names if name.startswith('xl/worksheets/sheet') and name.endswith('.xml'))
        if not worksheet_names:
            return pd.DataFrame()

        sheet = ET.fromstring(archive.read(worksheet_names[0]))
        rows = []
        width = 0

        for row in sheet.findall('.//main:sheetData/main:row', EXCEL_NAMESPACE):
            values = []
            for cell in row.findall('main:c', EXCEL_NAMESPACE):
                column_index = excel_column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column_index:
                    values.append(np.nan)

                cell_type = cell.attrib.get('t')
                if cell_type == 'inlineStr':
                    value = ''.join(text.text or '' for text in cell.findall('.//main:t', EXCEL_NAMESPACE))
                else:
                    value_node = cell.find('main:v', EXCEL_NAMESPACE)
                    value = value_node.text if value_node is not None else np.nan
                    if cell_type == 's' and pd.notna(value):
                        value = shared_strings[int(value)]

                values[column_index] = value

            width = max(width, len(values))
            rows.append(values)

    if not rows:
        return pd.DataFrame()

    rows = [row + [np.nan] * (width - len(row)) for row in rows]
    return pd.DataFrame(rows[1:], columns=make_unique_columns(rows[0])).replace(r'^\s*$', np.nan, regex=True)


def read_data_file(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix not in SUPPORTED_EXTENSIONS:
        raise ValueError(f'Unsupported file extension: {path.suffix}')

    if suffix == '.csv':
        return pd.read_csv(path, sep=None, engine='python', encoding='utf-8-sig').replace(r'^\s*$', np.nan, regex=True)

    try:
        return pd.read_excel(path).replace(r'^\s*$', np.nan, regex=True)
    except ImportError:
        return read_excel_without_engine(path)


## 6. Load Raw Data


In [6]:
def load_raw_datasets(catalog):
    datasets = {}
    summary_rows = []

    for meta in catalog:
        frame = read_data_file(meta['path'])
        datasets[meta['dataset_id']] = frame
        summary_rows.append(
            {
                'dataset_id': meta['dataset_id'],
                'rows': len(frame),
                'columns': frame.shape[1],
                'domain': meta['domain'],
            }
        )

    return datasets, pd.DataFrame(summary_rows)


raw_datasets, raw_summary_df = load_raw_datasets(DATASET_CATALOG)
display(raw_summary_df)


,dataset_id,rows,columns,domain
0,ecommerce_2024_01_january,431,18,ecommerce_sales
1,ecommerce_2024_06_june,697,18,ecommerce_sales
2,ecommerce_2024_12_december,1214,17,ecommerce_sales
3,ecommerce_2025_02_february,957,18,ecommerce_sales
4,ecommerce_2025_07_july,766,17,ecommerce_sales
5,ecommerce_2025_11_november,1131,18,ecommerce_sales
6,daily_household_transactions,2461,8,household_finance
7,personal_finance,1500,5,personal_finance


## 7. Cleaning Constants


In [7]:
USD_TO_IDR = 17_544
INR_TO_IDR = 183

STANDARD_CATEGORIES = ['Makanan', 'Transportasi', 'Hiburan', 'Belanja', 'Pendidikan', 'Kesehatan', 'Tagihan', 'Lainnya']
HEDONIC_CATEGORIES = {'Hiburan', 'Belanja'}
NEUTRAL_CATEGORIES = {'Makanan', 'Transportasi', 'Lainnya'}
UTIL_CATEGORIES = {'Pendidikan', 'Kesehatan', 'Tagihan'}
DROP_CATEGORY_KEYS = {'ppf', 'equity_mutual_fund', 'mutual_fund', 'investment', 'savings', 'saving', 'insurance', 'tax'}

DIRECT_CATEGORY_MAP = {
    'food': 'Makanan',
    'food_and_drink': 'Makanan',
    'food_drink': 'Makanan',
    'grocery': 'Makanan',
    'groceries': 'Makanan',
    'restaurant': 'Makanan',
    'snacks': 'Makanan',
    'makanan': 'Makanan',
    'transportation': 'Transportasi',
    'transport': 'Transportasi',
    'travel': 'Transportasi',
    'commute': 'Transportasi',
    'train': 'Transportasi',
    'bus': 'Transportasi',
    'transportasi': 'Transportasi',
    'entertainment': 'Hiburan',
    'subscription': 'Hiburan',
    'movies': 'Hiburan',
    'gaming': 'Hiburan',
    'culture': 'Hiburan',
    'festivals': 'Hiburan',
    'tourism': 'Hiburan',
    'social_life': 'Hiburan',
    'hiburan': 'Hiburan',
    'shopping': 'Belanja',
    'apparel': 'Belanja',
    'clothing': 'Belanja',
    'beauty': 'Belanja',
    'grooming': 'Belanja',
    'household': 'Belanja',
    'gift': 'Belanja',
    'belanja': 'Belanja',
    'education': 'Pendidikan',
    'self_development': 'Pendidikan',
    'pendidikan': 'Pendidikan',
    'health': 'Kesehatan',
    'health_and_fitness': 'Kesehatan',
    'fitness': 'Kesehatan',
    'medical': 'Kesehatan',
    'kesehatan': 'Kesehatan',
    'utilities': 'Tagihan',
    'rent': 'Tagihan',
    'bills': 'Tagihan',
    'mobile_service_provider': 'Tagihan',
    'water': 'Tagihan',
    'internet': 'Tagihan',
    'electricity': 'Tagihan',
    'phone': 'Tagihan',
    'tagihan': 'Tagihan',
    'other': 'Lainnya',
    'unknown': 'Lainnya',
    'lainnya': 'Lainnya',
}

KEYWORD_CATEGORY_RULES = [
    ('Makanan', ['makanan', 'minuman', 'food', 'snack', 'grocery', 'restaurant']),
    ('Transportasi', ['transport', 'commute', 'travel', 'train', 'bus', 'ojek']),
    ('Hiburan', ['entertainment', 'subscription', 'movie', 'gaming', 'mainan', 'festival', 'netflix', 'culture']),
    (
        'Belanja',
        [
            'shopping', 'fashion', 'pakaian', 'apparel', 'beauty', 'aksesoris', 'plastik', 'wadah', 'rak',
            'celengan', 'nampan', 'tray', 'baskom', 'mangkok', 'lunch_box', 'rantang', 'saringan',
            'pintu', 'perkakas', 'seal', 'baut', 'roof', 'household',
        ],
    ),
    ('Pendidikan', ['education', 'school', 'course', 'book', 'buku']),
    ('Kesehatan', ['health', 'fitness', 'medical', 'obat', 'olahraga']),
    ('Tagihan', ['utilities', 'rent', 'bill', 'mobile', 'water', 'internet', 'electricity', 'phone', 'listrik', 'pulsa']),
]

COLUMN_CANDIDATES = {
    'date': ['date', 'waktu_pesanan_dibuat'],
    'amount': ['amount', 'total_pembayaran'],
    'category': ['category', 'product_categories'],
    'description': ['transaction_description', 'description', 'note', 'order_id'],
    'transaction_type': ['type', 'income_expense', 'income/expense'],
    'status': ['status_pesanan'],
    'payment_method': ['metode_pembayaran', 'mode'],
    'city': ['kota_kabupaten', 'city'],
    'province': ['provinsi', 'province'],
    'quantity': ['total_qty', 'quantity'],
    'order_id': ['order_id'],
}


## 8. Cleaning Helpers


In [8]:
def clean_key(value):
    text = str(value).strip().lower().replace('&', ' and ')
    text = ''.join(character if character.isalnum() else '_' for character in text)

    while '__' in text:
        text = text.replace('__', '_')

    return text.strip('_')


def map_category(value, domain=None):
    key = clean_key(value)

    if key in DIRECT_CATEGORY_MAP:
        return DIRECT_CATEGORY_MAP[key]

    for category, keywords in KEYWORD_CATEGORY_RULES:
        if any(keyword in key for keyword in keywords):
            return category

    return 'Belanja' if domain == 'ecommerce_sales' else 'Lainnya'


def find_column(frame, role):
    lookup = {clean_key(column): column for column in frame.columns}

    for candidate in COLUMN_CANDIDATES[role]:
        key = clean_key(candidate)
        if key in lookup:
            return lookup[key]

        for clean_column, original_column in lookup.items():
            if key in clean_column:
                return original_column

    return None


def parse_dates(series):
    values = series.astype('string').str.strip()

    try:
        return pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)
    except TypeError:
        return pd.to_datetime(values, errors='coerce', dayfirst=True)


def dates_from_period(period, index):
    if not period:
        return pd.Series(pd.NaT, index=index)

    start = pd.Period(period, freq='M').start_time
    end = pd.Period(period, freq='M').end_time.floor('min')
    offsets = np.linspace(0, int((end - start).total_seconds() // 60), len(index)).round().astype(int)
    return pd.Series([start + pd.Timedelta(minutes=int(offset)) for offset in offsets], index=index)


def parse_amount(series):
    clean = series.astype('string').str.replace(r'\s+', '', regex=True)
    clean = clean.str.replace(r'(?<=\d),(?=\d{3}(\D|$))', '', regex=True).str.replace(',', '.', regex=False)
    clean = clean.str.replace(r'[^0-9.\-]', '', regex=True)
    return pd.to_numeric(clean, errors='coerce')


def currency_multiplier(dataset_id):
    if 'personal_finance' in dataset_id:
        return USD_TO_IDR, 'USD_TO_IDR'
    if 'daily_household' in dataset_id:
        return INR_TO_IDR, 'INR_TO_IDR'
    return 1, 'IDR'


## 9. Clean One Dataset


In [9]:
def clean_dataset(frame, meta):
    dataset_id = meta['dataset_id']
    work = frame.copy()
    before_rows = len(work)

    cols = {role: find_column(work, role) for role in COLUMN_CANDIDATES}

    type_col = cols['transaction_type']
    if type_col and work[type_col].astype('string').str.lower().str.contains('expense', na=False).any():
        work = work.loc[work[type_col].astype('string').str.lower().eq('expense')].copy()

    status_col = cols['status']
    if status_col and work[status_col].astype('string').str.lower().str.contains('selesai', na=False).any():
        work = work.loc[work[status_col].astype('string').str.lower().str.contains('selesai', na=False)].copy()

    date_col = cols['date']
    timestamp = parse_dates(work[date_col]) if date_col else dates_from_period(meta['period'], work.index)
    if timestamp.isna().all() and meta['period']:
        timestamp = dates_from_period(meta['period'], work.index)

    amount_col = cols['amount']
    amount = parse_amount(work[amount_col]) if amount_col else pd.Series(np.nan, index=work.index)
    multiplier, currency = currency_multiplier(dataset_id)
    amount_idr = amount * multiplier

    category_col = cols['category']
    category_raw = work[category_col].astype('string').fillna('unknown') if category_col else pd.Series('unknown', index=work.index)
    keep_category = ~category_raw.map(lambda value: any(drop_key in clean_key(value) for drop_key in DROP_CATEGORY_KEYS))

    cleaned = pd.DataFrame(
        {
            'dataset_id': dataset_id,
            'dataset_name': meta['dataset_name'],
            'domain': meta['domain'],
            'timestamp': timestamp,
            'amount': amount_idr,
            'amount_original': amount,
            'currency': currency,
            'category_raw': category_raw,
            'category': [map_category(value, meta['domain']) for value in category_raw],
            'description': work[cols['description']].astype('string').fillna('') if cols['description'] else '',
            'payment_method': work[cols['payment_method']].astype('string').fillna('unknown') if cols['payment_method'] else 'unknown',
            'city': work[cols['city']].astype('string').fillna('unknown') if cols['city'] else 'unknown',
            'province': work[cols['province']].astype('string').fillna('unknown') if cols['province'] else 'unknown',
            'quantity': parse_amount(work[cols['quantity']]) if cols['quantity'] else np.nan,
            'order_id': work[cols['order_id']].astype('string').fillna('') if cols['order_id'] else '',
        }
    )

    cleaned = cleaned.loc[keep_category.values].dropna(subset=['timestamp', 'amount']).query('amount > 0').copy()
    cleaned['amount'] = pd.to_numeric(cleaned['amount'], errors='coerce').astype(float)
    cleaned['amount_original'] = pd.to_numeric(cleaned['amount_original'], errors='coerce').astype(float)

    if len(cleaned) >= 10:
        cleaned['amount'] = cleaned['amount'].clip(upper=float(cleaned['amount'].quantile(0.99)))

    cleaned = cleaned.drop_duplicates(subset=['timestamp', 'amount', 'category', 'description', 'order_id']).reset_index(drop=True)

    summary = {
        'dataset_id': dataset_id,
        'raw_rows': before_rows,
        'clean_rows': len(cleaned),
        'removed_rows': before_rows - len(cleaned),
        'retention_pct': round(len(cleaned) / before_rows * 100, 2) if before_rows else 0,
        'top_category': cleaned['category'].mode().iloc[0] if not cleaned.empty else None,
        'currency': currency,
    }
    return cleaned, summary


## 10. Run Cleaning


In [10]:
def run_cleaning(catalog, raw_data):
    datasets = {}
    summary_rows = []

    for meta in catalog:
        dataset_id = meta['dataset_id']
        cleaned, summary = clean_dataset(raw_data[dataset_id], meta)
        datasets[dataset_id] = cleaned
        summary_rows.append(summary)

    summary = pd.DataFrame(summary_rows).sort_values('dataset_id').reset_index(drop=True)
    combined = pd.concat(datasets.values(), ignore_index=True).sort_values('timestamp').reset_index(drop=True)
    return datasets, summary, combined


cleaned_datasets, cleaning_summary_df, df_cleaned = run_cleaning(DATASET_CATALOG, raw_datasets)
display(cleaning_summary_df)
display(df_cleaned.head())


,dataset_id,raw_rows,clean_rows,removed_rows,retention_pct,top_category,currency
0,daily_household_transactions,2461,2069,392,84.07,Makanan,INR_TO_IDR
1,ecommerce_2024_01_january,431,367,64,85.15,Belanja,IDR
2,ecommerce_2024_06_june,697,608,89,87.23,Belanja,IDR
3,ecommerce_2024_12_december,1214,1042,172,85.83,Belanja,IDR
4,ecommerce_2025_02_february,957,831,126,86.83,Belanja,IDR
5,ecommerce_2025_07_july,766,675,91,88.12,Belanja,IDR
6,ecommerce_2025_11_november,1131,772,359,68.26,Belanja,IDR
7,personal_finance,1500,1222,278,81.47,Tagihan,USD_TO_IDR


,dataset_id,dataset_name,domain,timestamp,amount,amount_original,currency,category_raw,category,description,payment_method,city,province,quantity,order_id
0,daily_household_transactions,Daily Household Transactions,household_finance,2015-01-01,10980.0,60.0,INR_TO_IDR,Transportation,Transportasi,share jeep - Place T to brc,Cash,unknown,unknown,<NA>,
1,daily_household_transactions,Daily Household Transactions,household_finance,2015-01-01,1830.0,10.0,INR_TO_IDR,Transportation,Transportasi,share auto - hospital to brc station,Cash,unknown,unknown,<NA>,
2,daily_household_transactions,Daily Household Transactions,household_finance,2015-01-01,1830.0,10.0,INR_TO_IDR,Food,Makanan,tea,Cash,unknown,unknown,<NA>,
3,daily_household_transactions,Daily Household Transactions,household_finance,2015-01-01,5490.0,30.0,INR_TO_IDR,Transportation,Transportasi,bus - brc to Place H,Cash,unknown,unknown,<NA>,
4,daily_household_transactions,Daily Household Transactions,household_finance,2015-01-01,3660.0,20.0,INR_TO_IDR,Transportation,Transportasi,share auto - Place H to Place T base,Cash,unknown,unknown,<NA>,


## 11. Validate Cleaning


In [11]:
def build_validation_frame(datasets):
    rows = []

    for dataset_id, frame in datasets.items():
        rows.append(
            {
                'dataset_id': dataset_id,
                'rows': len(frame),
                'missing_timestamp': int(frame['timestamp'].isna().sum()),
                'missing_amount': int(frame['amount'].isna().sum()),
                'non_positive_amount': int((frame['amount'] <= 0).sum()),
                'invalid_category': int((~frame['category'].isin(STANDARD_CATEGORIES)).sum()),
            }
        )

    return pd.DataFrame(rows)


def validate_cleaning(validation):
    issue_columns = ['missing_timestamp', 'missing_amount', 'non_positive_amount', 'invalid_category']
    issue_count = validation[issue_columns].sum().sum()

    if issue_count:
        raise AssertionError(f'Cleaning masih punya {issue_count} issue kritis')

    print('Validasi cleaning lolos')


validation_df = build_validation_frame(cleaned_datasets)
display(validation_df)
validate_cleaning(validation_df)


,dataset_id,rows,missing_timestamp,missing_amount,non_positive_amount,invalid_category
0,ecommerce_2024_01_january,367,0,0,0,0
1,ecommerce_2024_06_june,608,0,0,0,0
2,ecommerce_2024_12_december,1042,0,0,0,0
3,ecommerce_2025_02_february,831,0,0,0,0
4,ecommerce_2025_07_july,675,0,0,0,0
5,ecommerce_2025_11_november,772,0,0,0,0
6,daily_household_transactions,2069,0,0,0,0
7,personal_finance,1222,0,0,0,0


Validasi cleaning lolos


## 12. Visualisasi Cleaning (Interactive Altair)


In [12]:
def build_category_summary(cleaned_frame):
    return cleaned_frame.groupby(['dataset_id', 'category']).size().reset_index(name='rows')


def sample_cleaned_transactions(cleaned_frame, max_rows=5000, random_state=42):
    return cleaned_frame.sample(min(len(cleaned_frame), max_rows), random_state=random_state)


def plot_cleaning_results(summary, cleaned_frame):
    selection = alt.selection_point(fields=['dataset_id'], bind='legend')

    retention_chart = alt.Chart(summary).mark_bar().encode(
        x=alt.X('retention_pct:Q', title='Retention (%)'),
        y=alt.Y('dataset_id:N', sort='-x', title='Dataset'),
        color=alt.Color('dataset_id:N', legend=alt.Legend(title='Dataset')),
        opacity=alt.condition(selection, alt.value(1), alt.value(0.25)),
        tooltip=['dataset_id', 'raw_rows', 'clean_rows', 'removed_rows', 'retention_pct', 'top_category'],
    ).add_params(selection).properties(title='Retention Cleaning per Dataset', height=260)

    category_chart = alt.Chart(build_category_summary(cleaned_frame)).mark_bar().encode(
        x=alt.X('rows:Q', title='Jumlah Transaksi'),
        y=alt.Y('category:N', sort='-x', title='Kategori'),
        color=alt.Color('dataset_id:N', legend=None),
        opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
        tooltip=['dataset_id', 'category', 'rows'],
    ).add_params(selection).properties(title='Distribusi Kategori Setelah Cleaning', height=260)

    amount_chart = alt.Chart(sample_cleaned_transactions(cleaned_frame)).mark_circle(size=35, opacity=0.45).encode(
        x=alt.X('timestamp:T', title='Tanggal'),
        y=alt.Y('amount:Q', title='Amount IDR'),
        color=alt.Color('dataset_id:N', legend=None),
        tooltip=['dataset_id', 'timestamp:T', 'amount:Q', 'category'],
    ).interactive().properties(title='Sebaran Amount Setelah Cleaning', height=280)

    return (retention_chart | category_chart) & amount_chart


plot_cleaning_results(cleaning_summary_df, df_cleaned)


alt.VConcatChart(...)

## 13. Output Cleaning


In [13]:
print('Data siap dipakai: df_cleaned, cleaned_datasets, cleaning_summary_df, validation_df')


Data siap dipakai: df_cleaned, cleaned_datasets, cleaning_summary_df, validation_df


## Area Analisis Mandiri
Gunakan cell kosong di bawah untuk eksperimen cleaning atau analisis lanjutan setelah semua cell utama dijalankan.

Function dan variabel yang bisa dipakai ulang:
- `read_data_file(path)`: membaca file CSV/XLSX menjadi dataframe.
- `load_raw_datasets(DATASET_CATALOG)`: membaca semua dataset mentah dan membuat summary raw.
- `map_category(value, domain)`: mengubah kategori mentah menjadi kategori standar.
- `clean_dataset(frame, meta)`: membersihkan satu dataset dan menghasilkan dataframe clean + summary.
- `run_cleaning(DATASET_CATALOG, raw_datasets)`: membersihkan semua dataset dan membuat `df_cleaned`.
- `build_validation_frame(cleaned_datasets)`: membuat dataframe validasi hasil cleaning.
- `validate_cleaning(validation_df)`: menghentikan proses jika masih ada issue kritis.
- `plot_cleaning_results(cleaning_summary_df, df_cleaned)`: membuat visualisasi interaktif hasil cleaning.
- `df_cleaned`: dataframe gabungan yang siap dipakai di tahap berikutnya.
- `cleaned_datasets`: dictionary dataframe bersih per dataset.
